# Lesson 03 - Agentic Design Patterns

У овој лекцији истражујемо три основна шаблона дизајна за изградњу ефикасних AI агената:

1. **Jасне инструкције за агента** — Креирање прецизних, улогу дефинишућих упита који усмеравају понашање агента  
2. **Структурисани излаз са Pydantic моделима** — Обезбеђивање да агенти враћају предвидљиве, проверене податке  
3. **Агенти са једном одговорношћу** — Дизајнирање фокусиранских агената који сваки раде једну ствар добро

Применићемо сваки шаблон на сценарио **препоруке туристичке дестинације**, постепено градећи систем који може предлагати дестинације, проверавати доступност и разматрати логистику.


## Постављање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Образац 1: Јасна упутства за агента

Најупечатљивији образац је и најједноставнији: писање јасних, детаљних упутстава за вашег агента.

Добра упутства дефинишу:
- **Ко** је агент (персона и тон)
- **Шта** треба да ради (одговорности корак по корак)
- **Како** треба да се понаша (ограничења и стил)

Испод правимо агента консијержа за путовања са јасним упутствима која обликују сваки одговор који производи.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Шаблон 2: Структурирани излаз са Пидантик моделима

Слободан текст је користан за разговор, али системи у даљем току процеса захтевају структуриране податке.  
Поређивањем **Пидантик модела** са **функцијом алата**, можемо:

- Дефинисати тачан шема за излаз агента  
- Аутоматски валидацију одговора  
- Поуздано интегрисати резултате агента у логику апликације  

Такође уводимо алат који враћа детаље одредишта како би агент темељио своје препоруке на стварним подацима.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Образац 3: Агент са једном одговорношћу

Комплексни задаци имају користи од расподеле посла између више фокусиранх агената, од којих сваки има једну одговорност:

- **Стручњак за дестинације** који зна о местима и доступности
- **Планер логистике** који се бави летовима, хотелима и маршрутама

Ово одговара принципу инжењеринга софтвера *раздвајања одговорности* — сваки агент је лакши за тестирање, одржавање и независно унапређење.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Резиме

У овој лекцији примењена су три агенцијска дизајнерска шаблона на сценарио препоруке путовања:

| Шаблон | Кључна идеја | Коришћеност |
|---|---|---|
| **Јасна упутства** | Одредити персонаж, одговорности и ограничења унапред | Конзистентно, у складу са брендом понашање агента |
| **Структурирани излаз** | Користити Pydantic моделе као формат одговора | Валидациони, машински читљиви резултати |
| **Једна одговорност** | Доделити сваком агенту један фокусирани задатак | Лакше тестирање, одржавање и састављање |

Ови шаблони се природно састављају — можете комбиновати јасна упутства са структурираним излазом унутар агента са једном одговорношћу како бисте изградили робусне системе спремне за производњу.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:  
Овај документ је преведен помоћу АИ сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако настојимо да буде прецизан, молимо вас да имате у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоруучује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из употребе овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
